# 🧠 How Does an LLM Actually Work?
### Live demos for the lecture — no API key needed, everything runs for free inside Colab

1. **Tokens**: how the AI chops up text
2. **Vectors**: the AI's "map of meaning"
3. **Context**: which *Harry* do you mean?
4. **Attention**: words looking at each other
5. **Next-word prediction**: the giant guessing machine

> Run the setup cell first (it takes about 1–2 minutes the first time, because it downloads the AI models).

### 🧭 How to use this notebook
- 📌 **First:** click **File → Save a copy in Drive** and work in the copy that opens in a new tab. Otherwise your changes won't be saved!
- A notebook is made of **cells**. Click a cell and press **Shift + Enter** to run it (or click the ▶️ button).
- Run the cells **in order, from top to bottom**.
- Cells marked ✏️ are for **you** to change. Edit the text between the quotes `"..."` and run the cell again.
- Cells with a title (like 🔧 Setup) hide boring code. Double-click them if you're curious what's inside!
- Something broken? **Runtime → Restart session and run all**.

In [ ]:
# @title 🔧 Setup (run me first!)
!pip install -q tiktoken sentence-transformers transformers bertviz > /tmp/pip.log 2>&1 || cat /tmp/pip.log

import os
import warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
warnings.filterwarnings("ignore")
from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()

import html
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import HTML, display
print("✅ Ready!")

---
## 1️⃣ Tokens: the AI doesn't read words, it reads *pieces*
Before an AI reads your message, it chops it into **tokens**. Each token gets an ID number.
This is the same tokenizer used by GPT-4o.

In [ ]:
# @title 🔧 Load the tokenizer
import tiktoken
encoder = tiktoken.get_encoding("o200k_base")
COLORS = ["#ffadad", "#ffd6a5", "#fdffb6", "#caffbf", "#9bf6ff", "#a0c4ff", "#bdb2ff", "#ffc6ff"]


def show_tokens(text):
    ids = encoder.encode(text)
    pieces = [encoder.decode_single_token_bytes(i).decode("utf-8", errors="replace") for i in ids]
    boxes = "".join(
        f'<span style="background:{COLORS[n % len(COLORS)]};color:black;padding:3px 2px;'
        f'margin:1px;border-radius:4px;font-size:22px;font-family:monospace">'
        f'{html.escape(p).replace(" ", "&nbsp;")}</span>'
        for n, p in enumerate(pieces))
    display(HTML(f"<div style='margin:8px 0'>{boxes} &nbsp;&nbsp;<b style='font-size:18px'>{len(ids)} tokens</b></div>"))
    print("   Token IDs:", ids)

In [ ]:
show_tokens("Harry Potter goes to Hogwarts.")
show_tokens("That's unbelievably supercalifragilisticexpialidocious!")
show_tokens("Καλημέρα! Τι κάνεις;")
show_tokens("I love pizza 🍕🍕🍕")

In [ ]:
# ✏️ Your turn: type anything and see how the AI chops it up
show_tokens("Type your own sentence here")

💡 **Notice:** common English words are usually **1 token**. Rare words, names and other languages get chopped into many pieces.
That's why AI used to struggle to count the letters in "strawberry": it never sees the letters, only the chunks!

---
## 2️⃣ Vectors: the AI's map of meaning
Every token becomes a long list of numbers called a **vector**. Think of it as an **address on a map of meaning**.
Words with similar meanings live close together.

In [ ]:
# @title 🔧 Load a small embedding model
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2")

vector = embedder.encode("cat")
print(f'The AI stores the word "cat" as a list of {len(vector)} numbers, like this:', np.round(vector[:5], 2), "...")
print("Too many numbers to imagine! So let's draw them as a map instead 👇")

In [ ]:
# @title 🗺️ Draw the map of meaning
WORD_GROUPS = {
    "animals":  ["cat", "dog", "lion", "tiger", "horse", "rabbit"],
    "food":     ["pizza", "pasta", "burger", "chocolate", "ice cream", "pancakes"],
    "sports":   ["football", "basketball", "tennis", "swimming", "volleyball"],
    "Hogwarts": ["Harry Potter", "Dumbledore", "Hogwarts", "magic wand", "wizard", "broomstick"],
    "tech":     ["computer", "robot", "smartphone", "internet", "video game"],
}


def draw_word_map(groups):
    words = [w for ws in groups.values() for w in ws]
    vecs = embedder.encode(words, normalize_embeddings=True)
    centred = vecs - vecs.mean(axis=0)
    _, _, vt = np.linalg.svd(centred, full_matrices=False)
    xy = centred @ vt[:2].T
    plt.figure(figsize=(11, 8))
    i = 0
    for group, ws in groups.items():
        pts = xy[i:i + len(ws)]
        plt.scatter(pts[:, 0], pts[:, 1], s=120, label=group)
        for (x, y), w in zip(pts, ws):
            plt.annotate(w, (x, y), xytext=(6, 4), textcoords="offset points", fontsize=13)
        i += len(ws)
    plt.legend(fontsize=12)
    plt.title("The map of meaning: similar words live close together", fontsize=15)
    plt.xticks([]); plt.yticks([])
    plt.show()


draw_word_map(WORD_GROUPS)

In [ ]:
# ✏️ Your turn: add your own group of words and redraw the map
WORD_GROUPS["music"] = ["guitar", "piano", "drums", "singer"]
draw_word_map(WORD_GROUPS)

---
## 3️⃣ Context: which *Harry* do you mean?
1. "**Harry** caught the Golden Snitch on his broomstick."
2. "Prince **Harry** visited the King at Buckingham Palace."
3. "**Harry** Styles sang his new song on stage."

Same word, three different people! At the start, the token "Harry" gets the **same vector** in every sentence.
Then **attention** mixes in information from the neighbouring words, and the vector *moves* on the map.

### 🔢 Before attention: the AI starts out confused
Let's ask a real language model (**BERT**, made by Google). First: what does "Harry" look like when the AI *starts* reading?

In [ ]:
# @title 🔧 Load BERT (a language model that is good at understanding context)
import re
from transformers import AutoTokenizer, AutoModel
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased", attn_implementation="eager")
bert.eval()

HARRYS = {
    "wizard": ["Harry caught the Golden Snitch on his broomstick.",
               "Harry waved his wand and cast a spell on the troll.",
               "Harry and Ron ran to Hagrid's hut after their potions lesson."],
    "prince": ["Prince Harry visited the King at Buckingham Palace.",
               "Prince Harry wore his royal uniform at the ceremony.",
               "Harry and his brother William waved to the crowd from the palace balcony."],
    "singer": ["Harry Styles sang his new song on stage.",
               "Harry Styles played the guitar at his concert.",
               "Harry's new album went straight to number one in the music charts."],
}
COLORS_BY_GROUP = ["#e8453c", "#1a73e8", "#34a853", "#f5b400"]
SURENESS = 15


def word_in_context(sentence, word):
    """The vector BERT gives to `word` inside `sentence` (after attention has done its job)."""
    found = re.search(rf"\b{re.escape(word)}(e?s)?\b", sentence, flags=re.IGNORECASE)
    if not found:
        raise ValueError(f'I can\'t find the word "{word}" in: "{sentence}". Check the spelling!')
    start, end = found.span()
    enc = bert_tok(sentence, return_tensors="pt", return_offsets_mapping=True)
    offsets = enc.pop("offset_mapping")[0].tolist()
    positions = [i for i, (s, e) in enumerate(offsets) if e > s and s < end and e > start]
    with torch.no_grad():
        hidden = bert(**enc).last_hidden_state[0]
    return torch.nn.functional.normalize(hidden[positions].mean(dim=0), dim=0)


def same_start(word, sentences):
    token_ids = bert_tok.convert_tokens_to_ids(bert_tok.tokenize(word))
    token_text = " + ".join(str(t) for t in token_ids)
    for sentence in sentences:
        marked = html.escape(sentence).replace(
            word, f'<b style="background:#ffe08a;color:black;border-radius:5px;padding:0 5px">{word}</b>')
        display(HTML(f'<div style="font-size:21px;margin:8px 0">{marked} &nbsp;→&nbsp; '
                     f'<b>{word}</b> = token <b>{token_text}</b></div>'))
    print(f'\n🤔 At the start, "{word}" is the SAME token (and the same vector) in every sentence.')
    print("   The AI can't tell them apart yet. Attention has to fix that!")


def examples_for(word, examples):
    if examples is not None:
        return examples
    if word.lower() in WORD_EXAMPLES:
        return WORD_EXAMPLES[word.lower()]
    print(f'😕 "{word}" isn\'t in the word list yet. Pick a word from the list, or add it to WORD_EXAMPLES!')
    return None


def meaning_map(word, examples=None):
    examples = examples_for(word, examples)
    if examples is None:
        return
    groups = [g for g, sentences in examples.items() for _ in sentences]
    vecs = torch.stack([word_in_context(s, word) for sentences in examples.values() for s in sentences]).numpy()
    centred = vecs - vecs.mean(axis=0)
    xy = centred @ np.linalg.svd(centred, full_matrices=False)[2][:2].T
    plt.figure(figsize=(9, 6))
    for k, group in enumerate(examples):
        points = xy[[i for i, g in enumerate(groups) if g == group]]
        color = COLORS_BY_GROUP[k % len(COLORS_BY_GROUP)]
        plt.scatter(points[:, 0], points[:, 1], s=300, color=color, alpha=0.85)
        plt.annotate(f"{word} = {group}", (points[:, 0].mean(), points[:, 1].max()), textcoords="offset points",
                     xytext=(0, 20), ha="center", fontsize=16, fontweight="bold", color=color)
    plt.margins(0.25)
    plt.xticks([])
    plt.yticks([])
    plt.title(f'After attention: every dot is "{word}" in a different sentence', fontsize=15)
    plt.show()


def which_meaning(sentence, word, examples=None):
    examples = examples_for(word, examples)
    if examples is None:
        return
    vec = word_in_context(sentence, word)
    closeness = {group: torch.stack([word_in_context(s, word) for s in sentences]).matmul(vec).mean().item()
                 for group, sentences in examples.items()}
    scores = np.array(list(closeness.values()))
    sure = np.exp(SURENESS * (scores - scores.max()))
    sure = 100 * sure / sure.sum()
    best = max(closeness, key=closeness.get)
    print(f'❓ "{sentence}"')
    print(f'🤖 I think "{word}" here means: {best.upper()}!')
    fig, ax = plt.subplots(figsize=(8, 0.8 * len(examples) + 0.8))
    ax.barh([f"{word} = {g}" for g in examples], sure,
            color=["#34a853" if g == best else "#dadce0" for g in examples])
    for i, percent in enumerate(sure):
        ax.text(percent + 1.5, i, f"{percent:.0f}%", va="center", fontsize=14)
    ax.set_xlim(0, 115)
    ax.invert_yaxis()
    ax.set_xticks([])
    ax.tick_params(axis="y", labelsize=14, length=0)
    for side in ["top", "right", "bottom"]:
        ax.spines[side].set_visible(False)
    ax.set_title("How sure is the AI?", fontsize=14)
    plt.tight_layout()
    plt.show()


def which_harry(sentence):
    which_meaning(sentence, "Harry", HARRYS)

In [ ]:
same_start("Harry", [
    "Harry caught the Golden Snitch on his broomstick.",
    "Prince Harry visited the King at Buckingham Palace.",
    "Harry Styles sang his new song on stage.",
])

### 🗺️ After attention: the Harrys move apart
Now attention mixes in the neighbouring words. We put "Harry" from 9 different sentences on the map of meaning. Each dot is one sentence.
**Do the three kinds of Harry end up in different places?**

In [ ]:
meaning_map("Harry", HARRYS)

### 🎮 Game: Which Harry is it?
The AI reads a **mystery sentence** and decides which Harry it is, using only the neighbouring words.
**Guess first, then run the cell!**

In [ ]:
which_harry("Harry was sorted into Gryffindor by the talking hat.")

In [ ]:
which_harry("Fans screamed when Harry walked onto the stage with his microphone.")

In [ ]:
# ✏️ Your turn: write a sentence about one of the Harrys. Can you FOOL the AI by mixing clues?
which_harry("Harry flew over the palace on his broomstick.")

### 🔤 More words with two meanings
The same trick works for lots of words! Run the word list, pick a word, and write your own mystery sentence.
**Challenge:** can you write a sentence where the AI gets it WRONG?

In [ ]:
# @title 📖 Words with two meanings (double-click to see all the example sentences)
WORD_EXAMPLES = {
    "apple": {
        "fruit": ["I ate a juicy red apple for lunch.",
                  "She picked an apple from the tree in the garden.",
                  "Grandma cut the apple into slices for the fruit salad."],
        "company": ["Apple released a new iPhone today.",
                    "My dad bought an Apple laptop for work.",
                    "Apple opened a huge new store in the city centre."],
    },
    "bat": {
        "animal": ["The bat flew out of the dark cave at night.",
                   "A tiny bat was hanging upside down from the tree.",
                   "The bat uses echoes to find insects in the dark."],
        "sport": ["He hit the ball with his cricket bat.",
                  "She swung the baseball bat as hard as she could.",
                  "The player picked up his bat and walked onto the pitch."],
    },
    "mouse": {
        "animal": ["The cat chased a little grey mouse across the kitchen.",
                   "A mouse nibbled the cheese in the cupboard.",
                   "The tiny mouse squeaked and hid in its hole."],
        "computer": ["Click the left mouse button to open the file.",
                     "My mouse stopped working, so I can't move the cursor.",
                     "Gamers like a fast wireless mouse and a good keyboard."],
    },
    "bank": {
        "money": ["I put my birthday money in the bank.",
                  "The bank gave my parents a loan to buy a house.",
                  "She went to the bank to open a savings account."],
        "river": ["We had a picnic on the bank of the river.",
                  "The fisherman sat on the muddy bank all afternoon.",
                  "Ducks were sleeping on the grassy bank by the water."],
    },
    "star": {
        "sky": ["We saw a shooting star in the night sky.",
                "The Sun is the star closest to Earth.",
                "That bright star twinkled above the mountains."],
        "famous person": ["The movie star signed autographs for her fans.",
                          "He became a football star at only eighteen.",
                          "The pop star sang her biggest hit at the concert."],
    },
    "fan": {
        "machine": ["It was so hot that I turned on the fan.",
                    "The fan on the ceiling spun around and cooled the room.",
                    "My laptop fan gets loud when I play games."],
        "supporter": ["He is a huge fan of Real Madrid.",
                      "Every fan in the stadium cheered when the goal went in.",
                      "She is a big fan of Taylor Swift and knows every song."],
    },
    "amazon": {
        "rainforest": ["Jaguars and parrots live in the Amazon rainforest.",
                       "The Amazon is the biggest river in South America.",
                       "Scientists explored the Amazon by boat for weeks."],
        "shop": ["I ordered a new phone case from Amazon.",
                 "The Amazon delivery van brought our parcel this morning.",
                 "My mum buys books on Amazon because it's quick."],
    },
    "crane": {
        "bird": ["A white crane stood on one leg in the lake.",
                 "The crane spread its wings and flew over the marsh.",
                 "We saw a crane catching fish with its long beak."],
        "machine": ["The crane lifted the steel beams to the top of the building.",
                    "Builders used a tall crane to build the new tower.",
                    "The crane driver sits in a tiny cabin high above the street."],
    },
}

for word, meanings in WORD_EXAMPLES.items():
    print(f"🔤 {word:7} → {' or '.join(meanings)}?")

In [ ]:
# ✏️ Your turn: pick a word from the list, then write your OWN mystery sentence with it
which_meaning("The mouse ran under the sofa when it saw the cat.", "mouse")

In [ ]:
# ✏️ More mystery sentences to try (change them, or find one that FOOLS the AI!)
which_meaning("We sat on the bank and watched the boats go by.", "bank")
which_meaning("Everyone wanted a selfie with the star of the film.", "star")
which_meaning("The fan was so loud that I couldn't sleep.", "fan")

In [ ]:
# ✏️ Draw the map for any word in the list
meaning_map("bat")

In [ ]:
# ⭐ Bonus: add a brand-new word. Give 2 or more example sentences for each meaning
WORD_EXAMPLES["jaguar"] = {
    "animal": ["The jaguar hunted quietly in the jungle.", "A jaguar has golden fur with black spots."],
    "car": ["My uncle drives a shiny new Jaguar.", "The Jaguar sped down the motorway."],
}
which_meaning("The jaguar climbed a tree to sleep in the shade.", "jaguar")

---
## 4️⃣ Attention: words looking at each other
> "The trophy didn't fit in the suitcase because **it** was too **big**." → *it* = the trophy
> "The trophy didn't fit in the suitcase because **it** was too **small**." → *it* = the suitcase

When the model reads "it", **attention** lets it look back at every other word and decide which ones matter.

### 👀 Optional: a 30-second peek inside a real AI
Each line shows one word "looking at" another word. Hover over **it** on the left to see which words it looks at.
Don't worry about the details: real AIs like ChatGPT have *thousands* of these lines!

In [ ]:
# @title 🔧 Attention viewer
from bertviz import head_view


def show_attention(sentence):
    inputs = bert_tok(sentence, return_tensors="pt")
    with torch.no_grad():
        out = bert(**inputs, output_attentions=True)
    tokens = bert_tok.convert_ids_to_tokens(inputs["input_ids"][0])
    head_view(out.attentions, tokens)

In [ ]:
show_attention("The trophy didn't fit in the suitcase because it was too big.")

---
## 5️⃣ The giant guessing machine: predicting the next word
**GPT-2** is a small, older cousin of ChatGPT. It has 124 million numbers inside it. Modern models have hundreds of billions!
It does just ONE thing: given some text, it gives a **probability for every possible next token**.

In [ ]:
# @title 🔧 Load GPT-2
from transformers import AutoModelForCausalLM
gpt2_tok = AutoTokenizer.from_pretrained("gpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("gpt2")
gpt2.eval()


def next_word_probabilities(text):
    ids = gpt2_tok(text, return_tensors="pt").input_ids
    with torch.no_grad():
        logits = gpt2(ids).logits[0, -1]
    return logits


def show_next_words(text, top=10):
    probs = torch.softmax(next_word_probabilities(text), dim=-1)
    p, idx = probs.topk(top)
    words = [repr(gpt2_tok.decode([int(i)])) for i in idx]
    plt.figure(figsize=(9, 5))
    plt.barh(words[::-1], (p * 100).numpy()[::-1], color="#7fb3ff")
    for y, v in enumerate((p * 100).numpy()[::-1]):
        plt.text(v + 0.5, y, f"{v:.1f}%", va="center", fontsize=12)
    plt.title(f'"{text} ..."', fontsize=14)
    plt.xlabel("probability (%)", fontsize=12)
    plt.yticks(fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
show_next_words("The capital of France is")

In [ ]:
show_next_words("Harry Potter waved his magic")

In [ ]:
# ✏️ Your turn: have the class guess first, then reveal!
show_next_words("For breakfast I had a bowl of")

### ✍️ Writing one word at a time
To write a sentence, the model **picks one token, adds it to the text, and predicts again**, over and over.
It picks *randomly*, but likely words get picked more often, like rolling a weighted dice 🎲.

**Temperature** controls how adventurous it is:
- low (0.2): always picks the safest word, so it's boring but sensible
- high (1.5): takes big risks, so it's creative... or total nonsense

In [ ]:
# @title 🔧 Story writer
def write(text, tokens=12, temperature=0.8, top_k=20):
    print(f"🟦 Start: {text}\n")
    for step in range(1, tokens + 1):
        logits = next_word_probabilities(text) / temperature
        top_logits, top_ids = logits.topk(top_k)
        probs = torch.softmax(top_logits, dim=-1)
        choice = torch.multinomial(probs, 1).item()
        token = gpt2_tok.decode([int(top_ids[choice])])
        print(f"   step {step:2d}: picked {token!r:15} (chance: {probs[choice] * 100:4.1f}%)")
        text += token
    print(f"\n📖 Result: {text}")

In [ ]:
write("Once upon a time, in a castle full of wizards,", temperature=0.8)

In [ ]:
# ✏️ Run this cell several times. Do you get the same story? Then try temperature=0.2 and temperature=1.5
write("Once upon a time, in a castle full of wizards,", temperature=1.5)

---
## 🎯 The big idea
> **An LLM is a very large statistical machine that predicts the next word based on the context.**

1. Text → **tokens**
2. Tokens → **vectors** (positions on the map of meaning)
3. **Attention** mixes in context ("which Harry?")
4. Predict the **probability** of every possible next token
5. Pick one, add it, repeat!

It never "looks up" facts. It picks words that *sound* right. That's why it can sound confident and still be wrong (**hallucination**).